In [1]:
import numpy as np
import pyvista as pv
from pyau3d.files import TopFile, PartFile, ModeFile
from pyau3d.utils import PltFileUtils, NodFileUtils, AnimSmartStream, GrpFileUtils
from pyau3d.pv.loaders import GrpFileVTK
from pyau3d.pv.loader.grpfilevtk import _grp2pv
from pyau3d.pv.vtktool.pdata import polydata
import vtk
import matplotlib.pyplot as plt
from matplotlib import cm, ticker
from pyau3d.constants import Constants
from scipy.signal import hilbert
import time
pv.set_jupyter_backend('trame')  # enables interactive in-notebook rendering

In [2]:
# specify group for animation file
# group = 2 corresponds to cylinder wall - could double check in .an02 file
RE = 60
Mach = 0.2
mesh_ver = 3
GROUP = 1
Mesh = 55222
ianimgrp = GROUP

# cfd_dir = f"/home/ahf25/CFD_2d_cylinder_all/Unsteady/M{Mach}/v{mesh_ver}_mesh/2d_cylinder_{Mesh}_Re{RE}_unsteady"

cfd_dir = f"/mnt/data1/ahf25/run2/2d_cylinder_{Mesh}_Re{RE}_unsteady"
nod_file = f"cylinder.nod"
plt_file = "cylinder.plt"
grp_file =f"cylinder.grp{GROUP:02}"
an_file = f"cylinder.an{GROUP:02}"
part_file = "cylinder.part"
top_file = "cylinder.top"

# specify file paths

path2nod = cfd_dir + "/" + nod_file
path2plt = cfd_dir  + "/" +  plt_file
path2grp = cfd_dir  + "/" +  grp_file
path2an = cfd_dir  + "/" +  an_file
path2part = cfd_dir  + "/" +  part_file
path2top = cfd_dir  + "/" +  top_file
 
mesh = PltFileUtils(path2plt)
nod = NodFileUtils(path2nod)
grp = GrpFileUtils(path2grp,GROUP)
groupfile = GrpFileVTK(path2grp,GROUP)
anim_vars = nod.ivars[GROUP - 1]
part_file = PartFile(path2part)
top = TopFile(path2top)

anim = AnimSmartStream(
    path2an,
    anim_vars=anim_vars,
    part=part_file,
    grp=grp
)

In [3]:
# create surface pv object from group file (file that stores surface geometry)
mesh = groupfile.transformtopv()

# define delta t (for the solver)
deltat = top["dtstr"] / Constants()["u"]

# define delta_t per frame IMPORTANT
frame_rate = 1 # output rate of animation  e.g frame_rate = 10 referes to 1 output per 10 iterations
deltat_frame = deltat * frame_rate # delta t between each frame

# read frames
# define time_selection
start = 1 # start of frame
end = anim.nframes  # end of frame
interval = 100  # internal of this code, does not mean the interval in the animation file

frames_selection = np.arange(start, end, interval)

# ===========================
# = Parameter numbers       =
# ===========================
# Maps parameter number -> (variable name, plot title)
PARAM_MAP = {
    1:  "x-coordinate",
    2:  "y-coordinate",
    3:  "z-coordinate",
    4:  "Density",
    5:  "U-velocity",
    6:  "V-velocity",
    7:  "W-velocity",
    8:  "Internal Energy",
    9:  "Pressure",
    10: "Mach Number",
    11: "Local Work",
}

# ---- CHOOSE YOUR PARAMETER HERE (just change this number) ----
PARAM_NUMBER = 5   # e.g. 5 = u-velocity, 9 = pressure, 10 = Mach number
# ----------------------------------------------------------------

var_name = PARAM_MAP[PARAM_NUMBER]
var_selection = np.array([PARAM_NUMBER]) - 1  # var_selection is 0-index in pyau3d

# read frames
test = anim.read_frames(frames_selection, var_selection)

# save the shape of test
n_frames, n_nodes, n_values = test.shape

print("Start of frames: ", start)
print("End of frames: ", end)
print("Interval: ", interval)
print(f"Frame output rate = 1 frame per {frame_rate} iterations")
print(f"Total physical time: {end * deltat_frame}s")
print(f"Viewing parameter: {var_name} (param #{PARAM_NUMBER})")

[SmartStream] Reading Parts:  66%|██████▌   | 25/38 [56:34<29:25, 135.79s/it] 


KeyboardInterrupt: 

In [ ]:
# saving the target values to scalar
scalars = test[:, :, 0]  # (n_frames, n_nodes) — scalar case

plotter = pv.Plotter(notebook=True)
mesh["field"] = scalars[0]
plotter.add_mesh(mesh, scalars="field", cmap="RdBu",
                  clim=[scalars.min(), scalars.max()])
plotter.add_text(var_name, font_size=12)

def update_frame(frame):
    mesh["field"] = scalars[int(frame)]
    plotter.render()

plotter.add_slider_widget(
    update_frame,
    rng=[0, len(scalars) - 1],
    value=0,
    title="Frame",
    fmt="%.0f"
)
plotter.view_xy()
plotter.show()

Widget(value='<iframe src="http://localhost:42891/index.html?ui=P_0x7636c83a5cf0_1&reconnect=auto" class="pyvi…